In [2]:
# Install necessary libraries for fine-tuning and basic data handling
!pip install -qqq peft trl accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.0 MB/s eta 0:00:00


## Local Inference on GPU
Model page: https://huggingface.co/Oscilla/Phi-3.5-mini-instruct-mlx-4Bit

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Oscilla/Phi-3.5-mini-instruct-mlx-4Bit)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define quantization configuration for 4-bit loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model and tokenizer explicitly with quantization_config
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", quantization_config=quantization_config, device_map="auto", trust_remote_code=True)

# Pass loaded model and tokenizer to pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
messages = [
    {"role": "user", "content": "What is insect?"},
]
pipe(messages)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import json
from datasets import Dataset

file_path = '/content/socratic_train.jsonl'
processed_data = []
errors_found = []

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            processed_data.append(json.loads(line))
        except json.JSONDecodeError as e:
            errors_found.append(f"Error parsing line {i+1}: {e} -- Line content: {line.strip()}")

if errors_found:
    print("--- Errors found during JSONL parsing ---")
    for error in errors_found:
        print(error)
    print("-----------------------------------------")

# Create a Dataset from the successfully parsed data
raw_dataset = Dataset.from_list(processed_data)

print(f"Successfully loaded {len(raw_dataset)} examples.")
print("Sample of the raw Socratic dataset (first 5 examples, showing 'messages' field):")
for i, example in enumerate(raw_dataset.select(range(min(5, len(raw_dataset))))):
    print(f"Example {i+1}: {example['messages']}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint
import torch
import os
import re
import gc

from google.colab import drive
drive.mount("/content/drive")
# VM disk is wiped on restart. Checkpoints must live on Drive.
OUTPUT_DIR = "/content/drive/MyDrive/socratic_finetuned_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints:", OUTPUT_DIR)

# Set environment variable for CUDA memory management
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Define quantization configuration for 4-bit loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Clear GPU memory from previous sessions/models if any
if 'pipe' in globals():
    del pipe
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
torch.cuda.empty_cache()
gc.collect()

# Using a smaller model (Phi-3-mini) due to OutOfMemoryError with Mistral-7B
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Load the configuration first to handle potential rope_scaling issues
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

if hasattr(config, 'rope_scaling') and isinstance(config.rope_scaling, dict):
    if 'type' not in config.rope_scaling:
        config.rope_scaling = None

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    quantization_config=quantization_config,
    device_map={"": 0},
    trust_remote_code=True
)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Define a formatting function to convert 'messages' into a single string for training
def formatting_func(example):
    if not isinstance(example['messages'], list):
        raise ValueError(f"Expected 'messages' to be a list, but got {type(example['messages'])} for example: {example}")
    return {"text": tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}

# Define training arguments
def make_sft_config(**kwargs):
    dropped = []
    while True:
        try:
            cfg = SFTConfig(**kwargs)
            if dropped:
                print("SFTConfig ignored:", ", ".join(dropped))
            return cfg
        except TypeError as exc:
            m = re.search(r"unexpected keyword argument '([^']+)'", str(exc))
            if not m or m.group(1) not in kwargs:
                raise
            dropped.append(m.group(1))
            kwargs.pop(m.group(1))

training_arguments = make_sft_config(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit",
    save_steps=100,
    save_total_limit=3,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=False,
)

# Initialize SFTTrainer
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=raw_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_arguments,
    formatting_func=formatting_func,
)

print("SFTTrainer initialized. Ready to train!")
print("To start training, run: `sft_trainer.train()`")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Run this before training if you did not run the model cell. Checkpoints: Drive/socratic_finetuned_model

### Start Fine-tuning

Now, let's start the fine-tuning process. This will train the model using your Socratic dataset and the LoRA configuration. This step can take a while depending on the dataset size and hardware resources.

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint
ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print("resume:", ckpt or "new run")
sft_trainer.train(resume_from_checkpoint=ckpt)

### Save the Fine-tuned LoRA Adapters

Once training is complete, we need to save the LoRA adapters. These adapters contain the learned Socratic style and are much smaller than the full model. They can be pushed to Hugging Face Hub or saved locally.

In [ ]:
sft_trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to {OUTPUT_DIR} (Google Drive)")

### Converting to GGUF Format

To get a GGUF file, you need to perform a few more steps:

1.  **Merge LoRA Adapters with the Base Model**: The LoRA adapters only contain the changes. You need to merge them back into the original `mistralai/Mistral-7B-Instruct-v0.2` model to create a full fine-tuned model.
2.  **Convert the Merged Model to GGUF**: Use the `convert.py` script from the `llama.cpp` project to convert the merged model into the GGUF format.

Here's how you can do it:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import os

# Define the output directory for the merged model
merged_model_dir = "./socratic_mistral_merged_model"

print("Loading base model...")
# Load the base model in 4-bit (or full precision if memory allows for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    return_dict=True,
    torch_dtype=torch.bfloat16, # Use bfloat16 for better compatibility and efficiency
    device_map="auto",
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

print("Loading LoRA adapters...")
# Load the saved LoRA adapters
model = PeftModel.from_pretrained(base_model, output_dir)

print("Merging LoRA adapters with base model...")
# Merge LoRA adapters into the base model
# This creates a single model that behaves like the fine-tuned model
model = model.merge_and_unload()

# Save the merged model and tokenizer
os.makedirs(merged_model_dir, exist_ok=True)
model.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to {merged_model_dir}")

### Performing GGUF Conversion

Now that the model is fine-tuned and merged, we can proceed with converting it to the GGUF format using `llama.cpp`. This involves cloning the `llama.cpp` repository, building it, and then using its conversion script.

**Note**: These steps involve shell commands and can be resource-intensive. Ensure you have sufficient RAM available.

In [ ]:
# 1. Clone llama.cpp and build it using CMake
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && mkdir build && cd build && cmake .. && cmake --build .

# 2. Install Python dependencies for conversion
!pip install -r llama.cpp/requirements.txt

In [ ]:
import os

merged_model_dir = "./socratic_mistral_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

output_gguf_f16_path = f"{merged_model_dir}/socratic-mistral-7b-merged-f16.gguf"

# Path to the convert.py script relative to the notebook's root
convert_script_path = "llama.cpp/convert.py"

# Check if the convert.py script exists
if not os.path.exists(convert_script_path):
    print(f"Error: '{convert_script_path}' not found. Please ensure 'llama.cpp' was cloned and built successfully in the previous step (cell 8b38aa3f).")
else:
    # Run the conversion script using its direct path
    !python3 {convert_script_path} {merged_model_dir} --outfile {output_gguf_f16_path} --outtype f16

    print(f"FP16 GGUF model saved to {output_gguf_f16_path}")

### Optional: Quantize the GGUF Model

To make the GGUF model smaller and more efficient for inference on various hardware, you can quantize it. Q4_K_M is a good general-purpose quantization type.

In [ ]:
# 4. Quantize the GGUF model to a smaller size (e.g., Q4_K_M)
# This step requires the 'llama.cpp' executables to be built.

import os
merged_model_dir = "./socratic_mistral_merged_model"
output_gguf_f16_path = f"{merged_model_dir}/socratic-mistral-7b-merged-f16.gguf"
output_gguf_q4_path = f"{merged_model_dir}/socratic-mistral-7b-merged-q4_K_M.gguf"

# Check if the f16 GGUF file exists before attempting to quantize
if os.path.exists(output_gguf_f16_path):
    !./llama.cpp/quantize {output_gguf_f16_path} {output_gguf_q4_path} q4_K_M
    print(f"Q4_K_M GGUF model saved to {output_gguf_q4_path}")
else:
    print(f"Error: FP16 GGUF file not found at {output_gguf_f16_path}. Please ensure the conversion step completed successfully.")

### Next: Convert to GGUF using `llama.cpp`

After merging, you'll need to use the `llama.cpp` tools to convert the saved model to GGUF. This typically involves cloning the `llama.cpp` repository, installing its dependencies, and running their conversion script.

```bash
# 1. Clone llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && make

# 2. Install Python dependencies for conversion
!pip install -r llama.cpp/requirements.txt

# 3. Convert the merged model to FP16 GGUF
# Replace 'PATH_TO_LLAMA_CPP' with the actual path if not running from /content/llama.cpp
!python llama.cpp/convert.py {merged_model_dir} --outfile {merged_model_dir}/socratic-mistral-7b-merged-f16.gguf --outtype f16

# 4. (Optional) Quantize the GGUF model to a smaller size (e.g., Q4_K_M)
# !llama.cpp/quantize {merged_model_dir}/socratic-mistral-7b-merged-f16.gguf {merged_model_dir}/socratic-mistral-7b-merged-q4_K_M.gguf q4_K_M
```

**Note**: The `llama.cpp` conversion and quantization steps can be resource-intensive and might require a significant amount of RAM. Ensure you have sufficient resources in your Colab environment (e.g., by using a high-RAM runtime). Replace `merged_model_dir` with the actual path (`./socratic_mistral_merged_model`) if you run the shell commands outside this notebook's context.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define quantization configuration for 4-bit loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", quantization_config=quantization_config, device_map="auto", trust_remote_code=True)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))